# Training Pipeline for Differentiated Thyroid Cancer Recurrence
This notebook replicates the training process described in the research paper *Machine learning for risk stratification of thyroid cancer patients: a 15‑year cohort study*. 

We evaluate multiple models (SVM, KNN, Decision Tree, Random Forest, ANN) on three feature subsets:
1. **Clinical/Demographic (No ATA Risk)**
2. **ATA Risk Only**
3. **All Features (Combined)**

Metric reported: Sensitivity, Specificity, PPV, NPV, Accuracy, AUC.


In [1]:
# %pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, recall_score, precision_score

pd.set_option('display.max_columns', None)


In [3]:
# Load preprocessed data
train_path = "./data/preprocessed/1_differentiated_thyroid_cancer_recurrence_train_preprocessed.csv"
test_path = "./data/preprocessed/1_differentiated_thyroid_cancer_recurrence_test_preprocessed.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

LABEL = "Recurred"

y_train = df_train[LABEL]
X_train_full = df_train.drop(columns=[LABEL])

y_test = df_test[LABEL]
X_test_full = df_test.drop(columns=[LABEL])

# Identify ATA Risk columns (OneHotEncoded)
ata_cols = [c for c in X_train_full.columns if "Risk" in c]
print("ATA Risk Columns:", ata_cols)

# Define Feature Sets
# 1. No ATA Risk
X_train_no_ata = X_train_full.drop(columns=ata_cols)
X_test_no_ata = X_test_full.drop(columns=ata_cols)

# 2. ATA Risk Only
X_train_ata = X_train_full[ata_cols]
X_test_ata = X_test_full[ata_cols]

# 3. All Features
X_train_all = X_train_full
X_test_all = X_test_full

datasets = {
    "No ATA": (X_train_no_ata, X_test_no_ata),
    "ATA Only": (X_train_ata, X_test_ata),
    "All Features": (X_train_all, X_test_all)
}

print(f"Train shapes: No ATA {X_train_no_ata.shape}, ATA {X_train_ata.shape}, All {X_train_all.shape}")


ATA Risk Columns: ['cat__Risk_High', 'cat__Risk_Intermediate', 'cat__Risk_Low']
Train shapes: No ATA (283, 48), ATA (283, 3), All (283, 51)
